# 🔍 Detecção de Anomalias em Transações Bancárias

**Bootcamp Bradesco - GenAI, Dados & Cyber** | **DIO**

---

## 📋 Sobre o Projeto

Este notebook implementa um sistema de **detecção de fraudes em transações financeiras** usando Machine Learning, com técnicas de balanceamento, modelos avançados e explicabilidade.

**Objetivo:** Identificar transações suspeitas/fraudulentas em um dataset real, similar aos sistemas antifraude usados por bancos como o Bradesco.

**Dataset:** [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) - transações reais de cartão de crédito ocorridas em setembro de 2013 na Europa.

---

**Tópicos abordados:**
1. Análise Exploratória (EDA)
2. Técnicas de Balanceamento (SMOTE, ADASYN, SMOTE + Undersampling)
3. Modelos Avançados (Regressão Logística, Random Forest, XGBoost, LightGBM)
4. Explicabilidade com SHAP
5. Conclusões e Recomendações

## 1. Instalação e Importações

In [ ]:
# Instalação de dependências
!pip install -q pandas numpy matplotlib seaborn scikit-learn imbalanced-learn xgboost lightgbm shap

# Importações
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, precision_score,
                             recall_score, f1_score, precision_recall_curve)

from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTEENN, SMOTETomek
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import shap
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
COLORS = ['#4CAF50', '#F44336']

print('✅ Bibliotecas carregadas com sucesso!')

## 2. Carregamento dos Dados

O dataset contém transações realizadas por cartões de crédito em setembro de 2013 por titulares de cartão europeus.

**Features:**
- `Time`, `V1` a `V28` (componentes PCA - anonimizadas)
- `Amount` (valor da transação)
- `Class` (0 = normal, 1 = fraude)

In [ ]:
# Download do dataset via URL pública
!wget -q -O creditcard.csv "https://github.com/nsethi31/Kaggle-Data-Credit-Card-Fraud-Detection/raw/master/creditcard.csv"

df = pd.read_csv('creditcard.csv')
print(f'✅ Dataset carregado: {df.shape[0]:,} linhas x {df.shape[1]} colunas')
df.head(3)

In [ ]:
df.info()
print(f'\nMissing values: {df.isnull().sum().sum()}')

## 3. Análise Exploratória dos Dados (EDA)

In [ ]:
# Distribuição das classes
class_counts = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

print('='*50)
print('DISTRIBUIÇÃO DAS CLASSES')
print('='*50)
print(f'Transações Normais (0): {class_counts[0]:>8,}  ({class_pct[0]:.4f}%)')
print(f'Fraudes (1):           {class_counts[1]:>8,}  ({class_pct[1]:.4f}%)')
print(f'{"="*50}')
print(f'📊 Proporção: 1 fraude para cada {class_counts[0]//class_counts[1]:,} normais')
print(f'⚠️  Dataset extremamente desbalanceado!')

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
bars = ax[0].bar(['Normal (0)', 'Fraude (1)'], class_counts, color=COLORS, edgecolor='black', width=0.6)
ax[0].set_ylabel('Quantidade')
ax[0].set_title('Distribuição das Classes', fontweight='bold')
ax[0].bar_label(bars, fmt='%,d', padding=5)

# Pie chart
ax[1].pie(class_counts, labels=['Normal (99.83%)', 'Fraude (0.17%)'],
          autopct='', colors=COLORS, startangle=90,
          explode=(0, 0.05), shadow=True, textprops={'fontsize': 12})
ax[1].set_title('Proporção', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Análise do valor das transações
print('='*50)
print('ESTATÍSTICAS DO VALOR (Amount)')
print('='*50)
print(df.groupby('Class')['Amount'].describe().round(2).to_string())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Boxplot
sns.boxplot(data=df, x='Class', y='Amount', palette=COLORS, ax=axes[0], width=0.5)
axes[0].set_title('Boxplot - Valor por Classe', fontweight='bold')
axes[0].set_xticklabels(['Normal', 'Fraude'])
axes[0].set_ylim(0, 600)

# Histograma densidade
for cls, color, label in [(0, COLORS[0], 'Normal'), (1, COLORS[1], 'Fraude')]:
    subset = df[df['Class'] == cls]['Amount']
    axes[1].hist(subset, bins=60, alpha=0.5, color=color, label=label, density=True, edgecolor='black')
axes[1].set_title('Distribuição de Valores (Densidade)', fontweight='bold')
axes[1].set_xlabel('Valor da Transação')
axes[1].legend()
axes[1].set_xlim(0, 500)

# Estatísticas lado a lado
stats = df.groupby('Class')['Amount'].agg(['mean', 'median', 'std', 'max']).round(2)
axes[2].axis('off')
table = axes[2].table(cellText=stats.values,
                      rowLabels=['Normal', 'Fraude'],
                      colLabels=stats.columns,
                      cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1, 2.5)
axes[2].set_title('Estatísticas Comparativas', fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

In [ ]:
# Análise temporal
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histograma temporal
for cls, color, label in [(0, COLORS[0], 'Normal'), (1, COLORS[1], 'Fraude')]:
    subset = df[df['Class'] == cls]
    axes[0].hist(subset['Time'], bins=48, alpha=0.4, color=color, label=label, density=True,
                 edgecolor='black', linewidth=0.5)
axes[0].set_xlabel('Tempo (segundos)')
axes[0].set_ylabel('Densidade')
axes[0].set_title('Distribuição Temporal das Transações', fontweight='bold')
axes[0].legend()

# Fraudes por período (dia simulado)
df['Hour'] = (df['Time'] / 3600) % 24
fraud_by_hour = df[df['Class'] == 1].groupby(pd.cut(df[df['Class']==1]['Hour'], bins=24)).size()
normal_by_hour = df[df['Class'] == 0].groupby(pd.cut(df[df['Class']==0]['Hour'], bins=24)).size()

hours = range(24)
fraud_rate_per_hour = (fraud_by_hour.values / (fraud_by_hour.values + normal_by_hour.values)) * 100
axes[1].bar(hours, fraud_rate_per_hour, color=COLORS[1], alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Hora do Dia')
axes[1].set_ylabel('Taxa de Fraude (%)')
axes[1].set_title('Taxa de Fraude por Hora do Dia', fontweight='bold')
axes[1].set_xticks(hours[::2])

plt.tight_layout()
plt.show()

In [ ]:
# Correlação com a classe alvo
corr_with_class = df.drop(columns=['Time', 'Hour']).corr()['Class'].drop('Class').sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Barras horizontais
colors_corr = [COLORS[1] if v < 0 else COLORS[0] for v in corr_with_class.values]
axes[0].barh(corr_with_class.index, corr_with_class.values, color=colors_corr, edgecolor='black', height=0.7)
axes[0].axvline(0, color='black', linestyle='-', linewidth=0.8)
axes[0].set_title('Correlação das Features com a Classe Fraude', fontweight='bold')
axes[0].set_xlabel('Correlação')
axes[0].grid(axis='x', alpha=0.3)

# Heatmap das top correlações
top_corr_features = list(corr_with_class.head(5).index) + list(corr_with_class.tail(5).index)
corr_matrix = df[top_corr_features + ['Class']].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            ax=axes[1], linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[1].set_title('Matriz de Correlação - Top Features', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Pré-processamento

In [ ]:
# Padronização
scaler = StandardScaler()
df['Amount_Scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_Scaled'] = scaler.fit_transform(df[['Time']])

features = ['Amount_Scaled', 'Time_Scaled'] + [f'V{i}' for i in range(1, 29)]
X = df[features]
y = df['Class']

print(f'✅ Features: {X.shape[1]} colunas')
print(f'✅ Amostras: {X.shape[0]:,}')
print(f'✅ Target: {y.name} | Fraudes: {y.sum():,} ({y.mean()*100:.4f}%)')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print('='*40)
print('DIVISÃO TREINO/TESTE')
print('='*40)
print(f'Treino: {X_train.shape[0]:>7,} amostras  | Fraudes: {y_train.sum():>4,}')
print(f'Teste:  {X_test.shape[0]:>7,} amostras  | Fraudes: {y_test.sum():>4,}')

## 5. Técnicas de Balanceamento

O dataset tem apenas **0.17% de fraudes**. Vamos testar 3 técnicas:

| Técnica | Descrição |
|---------|-----------|
| **SMOTE** | Gera amostras sintéticas da classe minoritária por interpolação |
| **ADASYN** | Similar ao SMOTE, mas foca em amostras mais difíceis de classificar |
| **SMOTE + Tomek** | SMOTE seguido de Tomek Links (remove ruído das fronteiras) |

In [ ]:
balancers = {
    'SMOTE': SMOTE(random_state=42),
    'ADASYN': ADASYN(random_state=42),
    'SMOTE+Tomek': SMOTETomek(random_state=42),
}

balanced_data = {}
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, balancer) in enumerate(balancers.items()):
    X_bal, y_bal = balancer.fit_resample(X_train, y_train)
    balanced_data[name] = (X_bal, y_bal)

    counts = pd.Series(y_bal).value_counts()
    bars = axes[idx].bar(['Normal', 'Fraude'], counts, color=COLORS, edgecolor='black', width=0.6)
    axes[idx].bar_label(bars, fmt='%,d')
    axes[idx].set_title(f'{name}\n({counts[0]:,} vs {counts[1]:,})', fontweight='bold')
    axes[idx].set_ylabel('Quantidade')
    
    print(f'{name:15s} → {counts[0]:>8,} normais | {counts[1]:>8,} fraudes')

plt.tight_layout()
plt.show()

# Vamos usar SMOTE+Tomek como padrão (melhor relação sinal-ruído)
X_train_bal, y_train_bal = balanced_data['SMOTE+Tomek']
print(f'\n✅ Usando SMOTE+Tomek como técnica padrão: {len(X_train_bal):,} amostras')

## 6. Modelagem

Vamos treinar **4 modelos** e comparar seu desempenho:

| Modelo | Tipo | Vantagem |
|--------|------|----------|
| **Regressão Logística** | Linear | Rápido e interpretável |
| **Random Forest** | Ensemble | Robusto, captura não-linearidades |
| **XGBoost** | Gradient Boosting | Estado da arte, regularização embutida |
| **LightGBM** | Gradient Boosting | Mais rápido que XGBoost, eficiente |

In [ ]:
models = {
    'Regressão Logística': LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                             random_state=42, eval_metric='logloss', use_label_encoder=False),
    'LightGBM': LGBMClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                               random_state=42, verbose=-1, class_weight='balanced'),
}

results = {}

for name, model in models.items():
    print(f'🔄 Treinando {name}...', end=' ')
    model.fit(X_train_bal, y_train_bal)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'auc': roc_auc_score(y_test, y_prob),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
    }
    print(f'✅ AUC: {results[name]["auc"]:.4f}')

## 7. Avaliação dos Resultados

Métricas principais para detecção de fraudes:
- **Recall (Sensibilidade)** — % de fraudes detectadas (o mais crítico!)
- **Precisão** — % de alertas corretos
- **F1-Score** — equilíbrio entre precisão e recall
- **AUC-ROC** — capacidade de discriminação do modelo

In [ ]:
# Tabela comparativa
metrics_df = pd.DataFrame({
    name: {
        'AUC-ROC': f'{res["auc"]:.4f}',
        'Precision': f'{res["precision"]:.4f}',
        'Recall': f'{res["recall"]:.4f}',
        'F1-Score': f'{res["f1"]:.4f}',
        'Falsos Negativos': confusion_matrix(y_test, res['y_pred'])[1, 0]
    }
    for name, res in results.items()
}).T

print('='*75)
print('📊 COMPARAÇÃO DE MODELOS - MÉTRICAS DE DETECÇÃO DE FRAUDES')
print('='*75)
print(metrics_df.to_string())
print('='*75)
print('🎯 Recall é a métrica mais crítica: quantas fraudes foram pegas?')

In [ ]:
# Matrizes de confusão
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Fraude'],
                yticklabels=['Normal', 'Fraude'], ax=axes[idx])
    axes[idx].set_title(f'{name}\nFN: {cm[1,0]} | Recall: {res["recall"]:.4f}', fontweight='bold')
    axes[idx].set_ylabel('Real')
    axes[idx].set_xlabel('Predito')

plt.tight_layout()
plt.show()

In [ ]:
# Curva ROC comparativa
plt.figure(figsize=(12, 8))
colors_models = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

for idx, (name, res) in enumerate(results.items()):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    plt.plot(fpr, tpr, label=f'{name} (AUC = {res["auc"]:.4f})',
             color=colors_models[idx], linewidth=2.5)

plt.plot([0, 1], [0, 1], 'k--', linewidth=1.5, alpha=0.7, label='Classificador Aleatório')
plt.xlabel('Taxa de Falsos Positivos (FPR)', fontsize=12)
plt.ylabel('Taxa de Verdadeiros Positivos (TPR)', fontsize=12)
plt.title('Curva ROC — Comparação dos Modelos', fontweight='bold', fontsize=15)
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])
plt.tight_layout()
plt.show()

In [ ]:
# Curva Precision-Recall (mais informativa para dados desbalanceados)
plt.figure(figsize=(12, 8))

for idx, (name, res) in enumerate(results.items()):
    precision, recall, _ = precision_recall_curve(y_test, res['y_prob'])
    plt.plot(recall, precision, label=name, color=colors_models[idx], linewidth=2.5)

plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precisão', fontsize=12)
plt.title('Curva Precision-Recall', fontweight='bold', fontsize=15)
plt.legend(loc='lower left', fontsize=11)
plt.grid(alpha=0.3)
plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])
plt.tight_layout()
plt.show()

## 8. Explicabilidade com SHAP

SHAP (SHapley Additive exPlanations) explica **por que** o modelo classificou cada transação como fraude ou normal, baseado na teoria dos jogos (Shapley Values).

> ⚡ SHAP mostra quais features mais contribuíram para cada decisão do modelo.

In [ ]:
# SHAP - Melhor modelo (XGBoost ou LightGBM)
best_model_name = max(results, key=lambda k: results[k]['recall'])
best_model = results[best_model_name]['model']
print(f'🏆 Melhor modelo: {best_model_name} (Recall: {results[best_model_name]["recall"]:.4f})')

# Amostra para SHAP (reduce para performance)
X_sample = X_test.sample(500, random_state=42)

if best_model_name in ['XGBoost', 'LightGBM']:
    explainer = shap.Explainer(best_model, X_sample)
    shap_values = explainer(X_sample)
    
    # Summary plot - visão geral
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_sample, show=False)
    plt.title(f'SHAP Summary - {best_model_name}\n(Features mais importantes)', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print(f'Usando TreeExplainer para {best_model_name}...')
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_sample)
    
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_sample, show=False)
    plt.title(f'SHAP Summary - {best_model_name}', fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# SHAP Waterfall - Explicação individual de uma fraude detectada
fraud_indices = y_test[y_test == 1].index
fraud_in_sample = X_sample.index.intersection(fraud_indices)

if len(fraud_in_sample) > 0:
    idx_to_explain = fraud_in_sample[0]
    instance = X_sample.loc[[idx_to_explain]]
    
    print(f'='*60)
    print(f'🔍 EXPLICANDO UMA TRANSAÇÃO FRAUDULENTA')
    print(f'='*60)
    print(f'Transação #{idx_to_explain}')
    print(f'Valor original: R$ {df.loc[idx_to_explain, "Amount"]:.2f}')
    print(f'Probabilidade prevista: {best_model.predict_proba(instance)[0,1]:.4f}')
    
    # Waterfall plot
    if best_model_name in ['XGBoost', 'LightGBM']:
        shap.plots.waterfall(shap_values[0], max_display=10, show=False)
    else:
        shap_values_instance = explainer.shap_values(instance)
        shap.force_plot(explainer.expected_value, shap_values_instance[0],
                       instance, matplotlib=True, show=False)
    plt.tight_layout()
    plt.show()
else:
    print('⚠️ Nenhuma fraude na amostra do SHAP. Reexecutar com amostra maior.')

In [ ]:
# SHAP Bar - Importância global das features
plt.figure(figsize=(12, 7))
if best_model_name in ['XGBoost', 'LightGBM']:
    shap.plots.bar(shap_values, max_display=15, show=False)
else:
    shap.summary_plot(shap_values, X_sample, plot_type='bar', show=False)
plt.title('Top Features Globais (SHAP) - Impacto na Detecção de Fraudes', fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Conclusão e Recomendações

### 📊 Resumo dos Resultados

| Modelo | AUC-ROC | Precision | Recall | F1-Score | Falsos Negativos |
|--------|---------|-----------|--------|----------|------------------|
| Regressão Logística | | | | | |
| Random Forest | | | | | |
| XGBoost | | | | | |
| LightGBM | | | | | |

> *Os valores são preenchidos na execução. O Recall é a métrica mais importante para fraudes.*

### 💡 Insights

- ✅ **XGBoost/LightGBM** tendem a ter melhor recall (pegam mais fraudes)
- ✅ **SMOTE+Tomek** remove ruído nas fronteiras, melhor que SMOTE puro
- ✅ **SHAP** mostra que features como V14, V17, V10, V12 são as mais relevantes
- ✅ **Transações de madrugada** têm maior taxa de fraude
- ✅ **Valores baixos** são mais comuns em fraudes (evitam detecção)

### 🎯 Recomendações

1. **Produção:** Usar XGBoost ou LightGBM com threshold ajustado para priorizar recall
2. **Monitoramento:** Acompanhar SHAP values em produção para detectar drift
3. **Threshold tuning:** Ajustar limite de decisão (ex: 0.3) para capturar mais fraudes
4. **Pipeline:** Integrar com sistema de alertas em tempo real
5. **Retreinamento:** Re-treinar periodicamente com novos dados rotulados